# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
#! pip install -q schedule pytest
#! pip install black jupyter-black

In [2]:
#Ячейка использовалась для проверки PEP8.
#%load_ext jupyter_black 

In [3]:
import time
import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (20 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [4]:
def get_book_data(book_url: str) -> dict:
    """
    Парсит данные о книге со страницы каталога сайта Books to Scrape.

    Функция получает HTML-страницу книги, извлекает основную информацию (название, цену, рейтинг, наличие, описание)
    и дополнительные характеристики из таблицы Product Information.

    Args:
        url (str): URL-адрес страницы книги для парсинга

    Returns:
        Optional[Dict]: Словарь с данными о книге в следующем формате:
            {
                'title': str,           # Название книги
                'price': str,           # Цена
                'rating': str,          # Рейтинг (количество звезд)
                'availability': str,    # Информация о наличии
                'description': str,     # Описание книги
                'upc': str,             # UPC код
                'product_type': str,    # Тип продукта
                'price_excl_tax': str,  # Цена без налога
                'price_incl_tax': str,  # Цена с налогом
                'tax': str,             # Размер налога
                'number_reviews': str # Количество отзывов
            }
        Возвращает None в случае ошибки при загрузке страницы.
    """

    try:
        response = requests.get(book_url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")
        title = soup.find("h1").text.strip()
        price = soup.find("p", class_="price_color").text.strip()

        rating_element = soup.find("p", class_="star-rating")
        rating_classes = rating_element.get("class", [])
        rating = [cls for cls in rating_classes if cls != "star-rating"][0]

        availability = soup.find("p", class_="instock availability").text.strip()

        product_description = soup.find("div", id="product_description")
        if product_description:
            description = product_description.find_next_sibling("p").text.strip()
        else:
            description = "No description available"

        product_info = {}
        info_table = soup.find("table", class_="table table-striped")
        if info_table:
            rows = info_table.find_all("tr")
            for row in rows:
                header = row.find("th").text.strip()
                value = row.find("td").text.strip()
                product_info[header] = value
        book_data = {
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "description": description,
            "upc": product_info.get("UPC", ""),
            "product_type": product_info.get("Product Type", ""),
            "price_excl_tax": product_info.get("Price (excl. tax)", ""),
            "price_incl_tax": product_info.get("Price (incl. tax)", ""),
            "tax": product_info.get("Tax", ""),
            "number_reviews": product_info.get("Number of reviews", ""),
        }
        return book_data
    except requests.RequestException as e:
        print(f"Ошибка при загрузке страницы: {e}")
        return None
    except Exception as e:
        print(f"Ошибка при парсинге данных: {e}")
        return None

In [5]:
book_url = "http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': '£51.77',
 'rating': 'Three',
 'availability': 'In stock (22 available)',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe 

## Задание 2. Сбор данных обо всех книгах (20 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [6]:
def scrape_books(save_to_file: bool = False) -> list:
    """
    Собирает данные обо всех книгах со всех страниц каталога Books to Scrape.

    Args:
        save_to_file (bool): Флаг для сохранения результатов в файл.

    Returns:
        list: Список словарей с данными о всех книгах
    """

    base_url = "http://books.toscrape.com/"
    all_books_data = []
    page_number = 1

    while True:
        page_url = f"{base_url}catalogue/page-{page_number}.html"

        print(f"Парсинг страницы {page_number}: {page_url}")

        response = requests.get(page_url)

        response.raise_for_status()

        soup = BeautifulSoup(response.content, "html.parser")

        book_cards = soup.find_all("article", class_="product_pod")

        for i, card in enumerate(book_cards, 1):
            book_link_tag = card.find("h3").find("a")
            book_link = book_link_tag["href"]
            book_url = f"{base_url}catalogue/{book_link}"
            book_data = get_book_data(book_url)
            all_books_data.append(book_data)

        # Проверяем наличие следующей страницы
        next_button = soup.find("li", class_="next")
        if not next_button:
            break

        page_number += 1

    if save_to_file and all_books_data:
        save_books_to_file(all_books_data)

    return all_books_data


def save_books_to_file(books_data: list) -> None:
    """Сохраняет данные о книгах в файл"""
    try:
        with open("books_data.txt", "w", encoding="utf-8") as file:
            file.write(f"ОТЧЕТ О ПАРСИНГЕ\n")
            file.write(f"Всего книг: {len(books_data)}\n")
            file.write("=" * 80 + "\n\n")

            for i, book in enumerate(books_data, 1):
                file.write(f"КНИГА №{i}\n")
                file.write(f"Название: {book.get('title', 'N/A')}\n")
                file.write(f"Цена: {book.get('price', 'N/A')}\n")
                file.write(f"Рейтинг: {book.get('rating', 'N/A')}\n")
                file.write(f"Наличие: {book.get('availability', 'N/A')}\n")
                file.write(f"UPC: {book.get('upc', 'N/A')}\n")
                file.write(f"Тип: {book.get('product_type', 'N/A')}\n")
                file.write(f"Цена без налога: {book.get('price_excl_tax', 'N/A')}\n")
                file.write(f"Цена с налогом: {book.get('price_incl_tax', 'N/A')}\n")
                file.write(f"Налог: {book.get('tax', 'N/A')}\n")
                file.write(
                    f"Количество отзывов: {book.get('number_of_reviews', 'N/A')}\n"
                )
                file.write(f"Описание: {book.get('description', 'N/A')[:200]}...\n")
                file.write("-" * 80 + "\n\n")

        print(f"Данные сохранены в 'books_data.txt'")

    except Exception as e:
        print(f"Ошибка при сохранении: {e}")

In [7]:
# Проверка работоспособности функции
res = scrape_books(save_to_file=True)
print(type(res), len(res))  # и проверки

Парсинг страницы 1: http://books.toscrape.com/catalogue/page-1.html
Парсинг страницы 2: http://books.toscrape.com/catalogue/page-2.html
Парсинг страницы 3: http://books.toscrape.com/catalogue/page-3.html
Парсинг страницы 4: http://books.toscrape.com/catalogue/page-4.html
Парсинг страницы 5: http://books.toscrape.com/catalogue/page-5.html
Парсинг страницы 6: http://books.toscrape.com/catalogue/page-6.html
Парсинг страницы 7: http://books.toscrape.com/catalogue/page-7.html
Парсинг страницы 8: http://books.toscrape.com/catalogue/page-8.html
Парсинг страницы 9: http://books.toscrape.com/catalogue/page-9.html
Парсинг страницы 10: http://books.toscrape.com/catalogue/page-10.html
Парсинг страницы 11: http://books.toscrape.com/catalogue/page-11.html
Парсинг страницы 12: http://books.toscrape.com/catalogue/page-12.html
Парсинг страницы 13: http://books.toscrape.com/catalogue/page-13.html
Парсинг страницы 14: http://books.toscrape.com/catalogue/page-14.html
Парсинг страницы 15: http://books.tosc

## Задание 3. Настройка регулярной выгрузки (10 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [8]:
def run_scheduler():
    """Запускает планировщик для ежедневного выполнения в 19:00"""
    schedule.every().day.at("19:00").do(lambda: scrape_books(save_to_file=True))

    while True:
        schedule.run_pending()
        time.sleep(60)


# run_scheduler()

## Задание 4. Написание автотестов (15 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [9]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest ../tests/test_scraper.py

============================= test session starts ==============================
platform darwin -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0
rootdir: /Users/ekaterinavalova/Documents/MIPT_master/1_Semester/Python/HW/HW_3/books_scraper
plugins: anyio-4.7.0
collected 5 items                                                              

.....                                           [100%]

======================== 5 passed in 677.99s (0:11:17) =========================


## Задание 5. Оформление проекта на GitHub и работа с Git (35 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds_2025.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```